# PPC v1 — Hybrid Projection-Conditioned Query Refinement Network (Fixed)

**Target metrics (paper-publishable):** Chamfer < 2 mm · F@2mm > 0.60 · F@5mm > 0.95 · HD95 < 5 mm

## What changed from v0 and why

| Issue in v0 | Fix in v1 |
|---|---|
| `W_PROJ = 0.15` → proj loss dominated (weighted 3.6 vs Chamfer 11) | `W_PROJ = 0.02` — Chamfer is the primary signal |
| Repulsion `h=0.03` repelled all neighbours, smeared points | `h=0.012` — only repels genuinely clumped points |
| Scale hard floor `[40,40,60]` made small/large patients inconsistent | Proportional scale `extent×0.55 + [20,20,30]` |
| Per-stage delta capped at ±0.15 — points never reach bone | Delta widened to ±0.25; clamp only at final output |
| `FeatureLift` had zero depth-axis variation | Learned depth embedding gives U-Net a head start |
| `QueryInitializer` global offset capped at ±0.15 | Widened to ±0.20 |
| `MAX_EVAL_SAMPLES = 20` — unreliable statistics | Full val set evaluation every epoch |
| `W_OFFSET = 0.002` suppressed refinement stages early | `W_OFFSET = 0.0005` — stages free to move early |
| `W_AUX_OCC = 0.10` aux signal competed with Chamfer | `W_AUX_OCC = 0.05` — purely a regulariser |
| Augmentation: intensity only | + Random horizontal flip (consistent across AP/LP/P) |

**Everything else (architecture, data split, backbone) is identical to v0.**

Output: `./ppc_network_v1/`


In [ ]:
import os, sys, json, time, math, random, shutil, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import vtk

warnings.filterwarnings('ignore', category=FutureWarning)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Total VRAM: {total_gb:.1f} GB')
    fraction = min(50.0 / total_gb, 0.95)
    torch.cuda.set_per_process_memory_fraction(fraction, device=0)
    print(f'VRAM limit: {fraction * total_gb:.1f} GB ({fraction*100:.1f}%)')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

print(f'PyTorch: {torch.__version__}')


In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path('./data/Lumbar_Filtered_1037')
SPLIT_FILE  = DATA_ROOT / 'dataset_split.json'
PROJECT_DIR = Path('./ppc_network_v1')   # NEW folder — v0 untouched
CKPT_DIR    = PROJECT_DIR / 'checkpoints'
LOG_DIR     = PROJECT_DIR / 'logs'
RESULTS_DIR = PROJECT_DIR / 'results'
for d in [PROJECT_DIR, CKPT_DIR, LOG_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── MODEL ──────────────────────────────────────────────────────────────────────
IMG_SIZE         = 512
COARSE_VOL_SIZE  = 32
AUX_VOL_SIZE     = 48
N_POINTS         = 5120
ENC_CHANNELS     = 192
VOL_CHANNELS     = 128
DEC_CHANNELS     = 128
QUERY_DIM        = 256
N_REFINE_STAGES  = 3
PRETRAINED       = True

# ── TRAINING ───────────────────────────────────────────────────────────────────
BATCH_SIZE       = 2
NUM_WORKERS      = 4
EPOCHS           = 200
LR               = 1e-4
WEIGHT_DECAY     = 1e-5
WARMUP_EPOCHS    = 8
USE_AMP          = True
GRAD_CLIP        = 1.0

# ── LOSS WEIGHTS (v1 fix — projection no longer dominates) ────────────────────
W_CHAMFER        = 1.0    # primary 3-D signal — unchanged
W_REPULSION      = 0.01   # was 0.03; still prevents collapse
W_PROJ           = 0.02   # was 0.15  ← THE KEY FIX (weighted proj was 3.6× Chamfer)
W_AUX_OCC        = 0.05   # was 0.10; pure regulariser now
W_OFFSET         = 0.0005 # was 0.002; refinement stages free to move early

# ── AUGMENTATION ───────────────────────────────────────────────────────────────
AUG_INTENSITY    = 0.15   # ±15% intensity jitter (slightly wider)
AUG_FLIP_PROB    = 0.5    # horizontal flip applied consistently to AP+LP+P

# ── EVAL ───────────────────────────────────────────────────────────────────────
MAX_EVAL_SAMPLES = 103    # full val set — was 20, which was statistically unreliable

# ── CHECKPOINT ─────────────────────────────────────────────────────────────────
CKPT_PATH      = CKPT_DIR / 'latest_checkpoint.pth'
BEST_CKPT_PATH = CKPT_DIR / 'best_checkpoint.pth'
TRAINING_LOG   = LOG_DIR  / 'training_log.json'

print('='*72)
print('CONFIGURATION — PPC v1 (Fixed Hybrid)')
print('='*72)
cfg = dict(IMG_SIZE=IMG_SIZE, N_POINTS=N_POINTS, COARSE_VOL_SIZE=COARSE_VOL_SIZE,
           AUX_VOL_SIZE=AUX_VOL_SIZE, ENC_CHANNELS=ENC_CHANNELS,
           BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR,
           W_CHAMFER=W_CHAMFER, W_REPULSION=W_REPULSION,
           W_PROJ=W_PROJ, W_AUX_OCC=W_AUX_OCC, W_OFFSET=W_OFFSET,
           N_REFINE_STAGES=N_REFINE_STAGES)
for k, v in cfg.items():
    print(f'  {k:<20} = {v}')
print(f'  Project dir: {PROJECT_DIR}')


In [ ]:
def load_vtk_points(path):
    reader = vtk.vtkPolyDataReader()
    reader.SetFileName(str(path))
    reader.Update()
    poly = reader.GetOutput()
    n = poly.GetNumberOfPoints()
    return np.array([poly.GetPoint(i) for i in range(n)], dtype=np.float32)


def save_vtk_points(points, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    vp = vtk.vtkPoints()
    for p in points:
        vp.InsertNextPoint(float(p[0]), float(p[1]), float(p[2]))
    vc = vtk.vtkCellArray()
    for i in range(len(points)):
        vc.InsertNextCell(1)
        vc.InsertCellPoint(i)
    poly = vtk.vtkPolyData()
    poly.SetPoints(vp)
    poly.SetVerts(vc)
    w = vtk.vtkPolyDataWriter()
    w.SetFileName(str(output_path))
    w.SetInputData(poly)
    w.SetFileTypeToASCII()
    w.Write()
    return output_path


def load_drr_image(path, size=IMG_SIZE):
    img = Image.open(path).convert('L')
    if img.size != (size, size):
        img = img.resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0


def load_projection_matrix(path):
    with open(path, 'r') as f:
        vals = [float(v) for v in f.read().split()]
    return np.array(vals, dtype=np.float32).reshape(3, 4)


def load_split(split_path=SPLIT_FILE):
    with open(split_path, 'r') as f:
        d = json.load(f)
    return d['train'], d['val'], d['test']


def append_training_log(log_path, epoch_record):
    log = {'records': []}
    if log_path.exists():
        with open(log_path, 'r') as f:
            log = json.load(f)
    log['records'].append(epoch_record)
    tmp = log_path.with_suffix('.json.tmp')
    with open(tmp, 'w') as f:
        json.dump(log, f, indent=2)
    tmp.replace(log_path)


train_ids, val_ids, test_ids = load_split()
print(f'Split: train={len(train_ids)}  val={len(val_ids)}  test={len(test_ids)}')


In [ ]:
def points_to_local(points, center, scale):
    """World mm → local [-1,1] frame."""
    return (points - center[None]) / (scale[None] + 1e-6)


def local_to_world(points_local, center, scale):
    """Local [-1,1] → world mm. Accepts batched tensors (B,N,3)."""
    return points_local * scale[:, None, :] + center[:, None, :]


def compute_scale(gt_full):
    """Proportional scale: always relative to patient extent.
    Fixes v0 hard floor [40,40,60] which made large/small patients inconsistent.
    """
    mins = gt_full.min(axis=0)
    maxs = gt_full.max(axis=0)
    extent = (maxs - mins).astype(np.float32)
    # 0.55 gives ~10% margin so surface points stay well inside [-1,1]
    scale = extent * 0.55 + np.array([20., 20., 30.], dtype=np.float32)
    return np.maximum(scale, np.array([50., 50., 80.], dtype=np.float32))


def make_aux_occupancy(points_local, vol_size=AUX_VOL_SIZE, dilate=1):
    pts = np.clip((np.asarray(points_local, np.float32) + 1.0) * 0.5, 0.0, 0.999999)
    idx = np.clip(np.floor(pts * vol_size).astype(np.int32), 0, vol_size - 1)
    occ = np.zeros((vol_size, vol_size, vol_size), dtype=np.float32)
    for dx in range(-dilate, dilate + 1):
        for dy in range(-dilate, dilate + 1):
            for dz in range(-dilate, dilate + 1):
                x = np.clip(idx[:, 0] + dx, 0, vol_size - 1)
                y = np.clip(idx[:, 1] + dy, 0, vol_size - 1)
                z = np.clip(idx[:, 2] + dz, 0, vol_size - 1)
                occ[x, y, z] = 1.0
    return occ


def flip_projection_matrix_horizontal(P, img_size=IMG_SIZE):
    """Flip the projection matrix for a horizontal mirror of the DRR.
    Equivalent to multiplying by diag(-1,1,1) in pixel space and
    re-centering: u' = (img_size-1) - u.
    """
    F_mat = np.array([
        [-1,  0,  img_size - 1],
        [ 0,  1,  0           ],
        [ 0,  0,  1           ],
    ], dtype=np.float32)
    # P is (3,4): flipped_P = F_mat @ P
    return F_mat @ P


class LumbarDatasetv1(Dataset):
    """Same as v0 dataset with two improvements:
    1. Proportional scale (no hard floor inconsistency).
    2. Consistent horizontal flip augmentation across both DRRs and both Ps.
    """
    def __init__(self, patient_ids, data_root=DATA_ROOT, img_size=IMG_SIZE, augment=False):
        self.patient_ids = patient_ids
        self.data_root   = Path(data_root)
        self.img_size    = img_size
        self.augment     = augment
        self.img_norm    = transforms.Normalize(mean=[0.485], std=[0.229])

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid  = self.patient_ids[idx]
        pdir = self.data_root / pid

        drr_ap = load_drr_image(pdir / 'AP_0'  / 'drr_AP_0.png',  self.img_size)
        drr_lp = load_drr_image(pdir / 'LP_90' / 'drr_LP_90.png', self.img_size)
        P_ap   = load_projection_matrix(pdir / 'AP_0'  / 'P_AP_0.txt')
        P_lp   = load_projection_matrix(pdir / 'LP_90' / 'P_LP_90.txt')
        gt_full = load_vtk_points(pdir / 'gt_ppc.vtk')

        # ── coordinate frame ──────────────────────────────────────────────────
        center = gt_full.mean(axis=0).astype(np.float32)
        scale  = compute_scale(gt_full)                # v1: proportional, no hard floor
        gt_local_full = np.clip(points_to_local(gt_full, center, scale), -1.0, 1.0)

        # ── auxiliary occupancy (for coarse 3-D supervision) ─────────────────
        aux_occ = make_aux_occupancy(gt_local_full, AUX_VOL_SIZE, dilate=1)

        # ── subsample GT to N_POINTS ──────────────────────────────────────────
        n = len(gt_full)
        sel = np.random.choice(n, N_POINTS, replace=(n < N_POINTS))
        gt_world = gt_full[sel].astype(np.float32)
        gt_local = gt_local_full[sel].astype(np.float32)

        # ── augmentation ──────────────────────────────────────────────────────
        if self.augment:
            # intensity jitter — independent per view, realistic DRR variation
            for drr in [drr_ap, drr_lp]:
                jitter = 1.0 + (np.random.rand() * 2 - 1) * AUG_INTENSITY
                drr[:] = np.clip(drr * jitter, 0.0, 1.0)

            # horizontal flip — MUST be consistent: flip both DRRs and both Ps
            if np.random.rand() < AUG_FLIP_PROB:
                drr_ap = drr_ap[:, ::-1].copy()
                drr_lp = drr_lp[:, ::-1].copy()
                P_ap   = flip_projection_matrix_horizontal(P_ap, self.img_size)
                P_lp   = flip_projection_matrix_horizontal(P_lp, self.img_size)

        # ── to tensors ────────────────────────────────────────────────────────
        drr_ap_t = self.img_norm(torch.from_numpy(drr_ap).unsqueeze(0).float())
        drr_lp_t = self.img_norm(torch.from_numpy(drr_lp).unsqueeze(0).float())

        return {
            'drr_ap':       drr_ap_t,
            'drr_lp':       drr_lp_t,
            'P_ap':         torch.from_numpy(P_ap).float(),
            'P_lp':         torch.from_numpy(P_lp).float(),
            'gt_ppc_world': torch.from_numpy(gt_world).float(),
            'gt_ppc_local': torch.from_numpy(gt_local).float(),
            'aux_occ':      torch.from_numpy(aux_occ).float(),
            'center':       torch.from_numpy(center).float(),
            'scale':        torch.from_numpy(scale).float(),
            'patient_id':   pid,
        }


In [ ]:
train_ids, val_ids, test_ids = load_split()
_ds = LumbarDatasetv1(train_ids[:2], augment=True)
_b  = _ds[0]
print('Dataset sanity check:')
print(f'  drr_ap   : {_b["drr_ap"].shape}   dtype={_b["drr_ap"].dtype}')
print(f'  P_ap     : {_b["P_ap"].shape}')
print(f'  gt_world : {_b["gt_ppc_world"].shape}  min={_b["gt_ppc_world"].min():.1f}  max={_b["gt_ppc_world"].max():.1f} mm')
print(f'  gt_local : {_b["gt_ppc_local"].shape}  min={_b["gt_ppc_local"].min():.3f}  max={_b["gt_ppc_local"].max():.3f}')
print(f'  center   : {_b["center"]}')
print(f'  scale    : {_b["scale"]}')
print(f'  aux_occ  : {_b["aux_occ"].shape}  occupied={_b["aux_occ"].mean()*100:.2f}%')
del _ds, _b
print('OK')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ENCODER — ResNet18 backbone, preserves 32×32 spatial features
# Identical to v0.
# ─────────────────────────────────────────────────────────────────────────────
class Encoder2D(nn.Module):
    def __init__(self, in_channels=1, out_channels=ENC_CHANNELS, pretrained=PRETRAINED):
        super().__init__()
        base    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        new_c1  = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        if pretrained:
            with torch.no_grad():
                new_c1.weight[:] = base.conv1.weight.mean(dim=1, keepdim=True)
        base.conv1  = new_c1
        self.stem   = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.proj   = nn.Conv2d(256, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.proj(x)   # (B, ENC_CHANNELS, 32, 32)


# ─────────────────────────────────────────────────────────────────────────────
# FEATURE LIFT — 2-D feature map → 3-D volume
# v1 fix: add learned depth embedding so the 3-D U-Net starts with depth-axis
# variation instead of a flat repeated slice. Tiny cost (~C×D params).
# ─────────────────────────────────────────────────────────────────────────────
class FeatureLift(nn.Module):
    def __init__(self, in_channels=ENC_CHANNELS, out_channels=VOL_CHANNELS, depth=COARSE_VOL_SIZE):
        super().__init__()
        # Learnable depth-axis embedding: shape (1, C, depth, 1, 1)
        self.depth_embed = nn.Parameter(torch.randn(1, in_channels, depth, 1, 1) * 0.02)
        self.refine = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
            nn.Conv3d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
        )

    def forward(self, feat_2d):
        B, C, H, W = feat_2d.shape
        # Tile 2-D feature across depth axis, then add learned variation
        vol = feat_2d.unsqueeze(2).expand(B, C, COARSE_VOL_SIZE, H, W).contiguous()
        vol = vol + self.depth_embed   # broadcast over B, H, W
        return self.refine(vol)        # (B, VOL_CHANNELS, D, H, W)


# ─────────────────────────────────────────────────────────────────────────────
# BIPLANAR FUSION — fuse AP and LP volumes into one canonical volume
# Identical to v0 (permutation aligns AP→XZ and LP→YZ planes).
# ─────────────────────────────────────────────────────────────────────────────
class BiplanarFusion(nn.Module):
    def __init__(self, in_channels=VOL_CHANNELS, out_channels=VOL_CHANNELS):
        super().__init__()
        self.fuse = nn.Sequential(
            nn.Conv3d(in_channels * 2, out_channels, 1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
            nn.Conv3d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(8, out_channels), nn.GELU(),
        )

    def forward(self, ap_vol, lp_vol):
        ap_aligned = ap_vol.permute(0, 1, 4, 2, 3).contiguous()  # depth←W
        lp_aligned = lp_vol.permute(0, 1, 2, 4, 3).contiguous()  # depth←H
        return self.fuse(torch.cat([ap_aligned, lp_aligned], dim=1))


# ─────────────────────────────────────────────────────────────────────────────
# 3-D U-NET — refines the fused volume and produces aux occupancy
# Identical to v0.
# ─────────────────────────────────────────────────────────────────────────────
class Block3D(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, padding=1), nn.GroupNorm(8, out_c), nn.GELU(),
            nn.Conv3d(out_c, out_c, 3, padding=1), nn.GroupNorm(8, out_c), nn.GELU(),
        )
    def forward(self, x): return self.block(x)


class CoarseUNet3D(nn.Module):
    def __init__(self, in_channels=VOL_CHANNELS, feat_channels=DEC_CHANNELS):
        super().__init__()
        self.enc1       = Block3D(in_channels, 96)
        self.down1      = nn.Conv3d(96,  160, 2, stride=2)
        self.enc2       = Block3D(160, 160)
        self.down2      = nn.Conv3d(160, 224, 2, stride=2)
        self.bottleneck = Block3D(224, 224)
        self.up2        = nn.ConvTranspose3d(224, 160, 2, stride=2)
        self.dec2       = Block3D(320, 160)
        self.up1        = nn.ConvTranspose3d(160, 96,  2, stride=2)
        self.dec1       = Block3D(192, feat_channels)
        self.aux_head   = nn.Sequential(
            nn.Conv3d(feat_channels, feat_channels // 2, 3, padding=1),
            nn.GELU(),
            nn.Conv3d(feat_channels // 2, 1, 1),
        )

    def forward(self, x):
        e1  = self.enc1(x)
        e2  = self.enc2(self.down1(e1))
        b   = self.bottleneck(self.down2(e2))
        d2  = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1  = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        aux = F.interpolate(self.aux_head(d1),
                            size=(AUX_VOL_SIZE,) * 3,
                            mode='trilinear', align_corners=True)
        return d1, aux   # (B,feat,D,H,W), (B,1,48,48,48)


# ─────────────────────────────────────────────────────────────────────────────
# QUERY INITIALIZER — learnable offset from a uniform grid
# v1 fix: global offset scale 0.15→0.20 so anchors can reach off-centre bone.
# ─────────────────────────────────────────────────────────────────────────────
class QueryInitializer(nn.Module):
    def __init__(self, n_points=N_POINTS, feat_channels=DEC_CHANNELS):
        super().__init__()
        xs = np.linspace(-0.8,  0.8, 16)
        ys = np.linspace(-0.8,  0.8, 16)
        zs = np.linspace(-0.9,  0.9, 20)
        grid = np.stack(np.meshgrid(xs, ys, zs, indexing='ij'), -1).reshape(-1, 3).astype(np.float32)
        self.register_buffer('base_queries', torch.from_numpy(grid))
        self.global_head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
            nn.Linear(feat_channels, 256), nn.GELU(),
            nn.Linear(256, n_points * 3),
        )

    def forward(self, vol_feat):
        B      = vol_feat.shape[0]
        offset = self.global_head(vol_feat).view(B, N_POINTS, 3)
        # 0.20 (was 0.15) lets anchors travel further from their grid cell
        return torch.clamp(self.base_queries[None] + 0.20 * torch.tanh(offset), -1.0, 1.0)


# ─────────────────────────────────────────────────────────────────────────────
# PROJECTION HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def project_points(P, points_world, img_size=IMG_SIZE):
    """Project world-mm points through a 3×4 projection matrix.
    Returns pixel coords (uv), normalised coords in [-1,1] (uv_norm),
    and log depth.
    """
    B, N, _ = points_world.shape
    ones  = torch.ones(B, N, 1, device=points_world.device, dtype=points_world.dtype)
    homog = torch.cat([points_world, ones], dim=-1)          # (B,N,4)
    uvw   = torch.bmm(homog, P.transpose(1, 2))              # (B,N,3)
    z     = uvw[..., 2:3].clamp(min=1e-6)
    uv    = uvw[..., :2] / z                                 # pixel coords
    uv_norm = (uv / (img_size - 1.0)) * 2.0 - 1.0           # [-1,1]
    return uv, uv_norm, torch.log(z)


def sample_2d_features(feat_map, uv_norm):
    """Bilinearly sample (B,C,H,W) at (B,N,2) normalised coords → (B,N,C)."""
    grid = uv_norm.view(feat_map.shape[0], -1, 1, 2)
    samp = F.grid_sample(feat_map, grid, mode='bilinear',
                         align_corners=True, padding_mode='border')
    return samp.squeeze(-1).transpose(1, 2)


def sample_3d_features(vol_feat, points_local):
    """Trilinearly sample (B,C,D,H,W) at local[-1,1] coords → (B,N,C)."""
    grid = torch.stack([points_local[..., 2],
                        points_local[..., 1],
                        points_local[..., 0]], dim=-1)
    grid = grid.view(points_local.shape[0], -1, 1, 1, 3)
    samp = F.grid_sample(vol_feat, grid, mode='bilinear',
                         align_corners=True, padding_mode='border')
    return samp.squeeze(-1).squeeze(-1).transpose(1, 2)


# ─────────────────────────────────────────────────────────────────────────────
# REFINEMENT STAGE — projection-conditioned iterative update
# v1 fix: per-stage delta 0.15→0.25; clamp REMOVED between stages
#         so accumulated displacement can reach ±0.75 over 3 stages.
#         Final output of PPCNetv1.forward() is clamped once.
# ─────────────────────────────────────────────────────────────────────────────
class RefinementStage(nn.Module):
    def __init__(self, feat_2d=ENC_CHANNELS, feat_3d=DEC_CHANNELS, hidden=QUERY_DIM):
        super().__init__()
        in_dim  = feat_2d * 2 + feat_3d + 3 + 2 + 2   # same as v0
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, 3),
        )

    def forward(self, q_local, vol_feat, feat_ap, feat_lp, P_ap, P_lp, center, scale):
        q_world    = local_to_world(q_local, center, scale)
        _, uvn_ap, _ = project_points(P_ap, q_world)
        _, uvn_lp, _ = project_points(P_lp, q_world)
        f3d = sample_3d_features(vol_feat, q_local)
        fap = sample_2d_features(feat_ap, uvn_ap)
        flp = sample_2d_features(feat_lp, uvn_lp)
        x   = torch.cat([f3d, fap, flp, q_local, uvn_ap, uvn_lp], dim=-1)
        delta = 0.25 * torch.tanh(self.mlp(x))   # was 0.15 — wider movement per stage
        # NO clamp here — clamp only once at the end of the full forward pass
        return q_local + delta, {'delta': delta}


# ─────────────────────────────────────────────────────────────────────────────
# FULL NETWORK
# ─────────────────────────────────────────────────────────────────────────────
class PPCNetv1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder_ap  = Encoder2D()
        self.encoder_lp  = Encoder2D()
        self.lift_ap     = FeatureLift()
        self.lift_lp     = FeatureLift()
        self.fusion      = BiplanarFusion()
        self.coarse3d    = CoarseUNet3D()
        self.query_init  = QueryInitializer()
        self.stages      = nn.ModuleList([RefinementStage() for _ in range(N_REFINE_STAGES)])

    def forward(self, drr_ap, drr_lp, P_ap, P_lp, center, scale):
        # 2-D encoding
        feat_ap  = self.encoder_ap(drr_ap)    # (B,192,32,32)
        feat_lp  = self.encoder_lp(drr_lp)

        # 2-D → 3-D feature volumes
        vol_ap   = self.lift_ap(feat_ap)       # (B,128,32,32,32)
        vol_lp   = self.lift_lp(feat_lp)

        # Fuse and refine with 3-D U-Net
        fused             = self.fusion(vol_ap, vol_lp)
        vol_feat, aux_occ = self.coarse3d(fused)

        # Initialize 5120 query points from uniform grid + global offset
        q_local   = self.query_init(vol_feat)  # (B,5120,3)  in [-1,1]

        # Iterative projection-conditioned refinement
        stage_aux = []
        for stage in self.stages:
            q_local, aux = stage(q_local, vol_feat, feat_ap, feat_lp, P_ap, P_lp, center, scale)
            stage_aux.append(aux)

        # Single clamp at the end — NOT after each stage
        q_local  = torch.clamp(q_local, -1.0, 1.0)
        q_world  = local_to_world(q_local, center, scale)

        return {
            'pred_local':    q_local,
            'pred_world':    q_world,
            'aux_occ_logits': aux_occ,
            'stage_aux':     stage_aux,
        }


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

_test_model = PPCNetv1()
print(f'PPCNetv1 parameters: {count_params(_test_model)/1e6:.1f} M')
del _test_model


In [ ]:
# Quick forward-pass check before starting training
_m = PPCNetv1().to(device)
_m.eval()
with torch.no_grad():
    _drr  = torch.randn(2, 1, IMG_SIZE, IMG_SIZE, device=device)
    _P    = torch.eye(3, 4, device=device).unsqueeze(0).repeat(2, 1, 1)
    _c    = torch.zeros(2, 3, device=device)
    _s    = torch.tensor([[80., 80., 120.]] * 2, device=device)
    _out  = _m(_drr, _drr, _P, _P, _c, _s)
print('Forward pass OK')
print(f'  pred_world : {_out["pred_world"].shape}  '
      f'min={_out["pred_world"].min():.2f}  max={_out["pred_world"].max():.2f}')
print(f'  aux_occ    : {_out["aux_occ_logits"].shape}')
print(f'  n stages   : {len(_out["stage_aux"])}')
del _m, _drr, _P, _c, _s, _out
torch.cuda.empty_cache()


In [ ]:
def pairwise_sqdist(x, y):
    """Squared pairwise distances between two point clouds. (B,N,3)×(B,M,3)→(B,N,M)."""
    x2 = (x ** 2).sum(-1, keepdim=True)
    y2 = (y ** 2).sum(-1).unsqueeze(1)
    xy = torch.bmm(x, y.transpose(1, 2))
    return (x2 + y2 - 2 * xy).clamp_min(0.0)


def chamfer_loss(pred, gt):
    """Symmetric Chamfer distance in mm² (mean over points and batch)."""
    d2 = pairwise_sqdist(pred, gt)
    return 0.5 * (d2.min(dim=2)[0].mean() + d2.min(dim=1)[0].mean())


def repulsion_loss(points, k=8, h=0.012):
    """Push k nearest neighbours apart in local [-1,1] space.
    v1 fix: h=0.012 (was 0.03).
    With 5120 points in a unit cube, mean NN dist ≈ 0.04, so h=0.03 repelled
    nearly every pair. h=0.012 targets only genuinely clumped points.
    """
    d2  = pairwise_sqdist(points, points)
    B, N, _ = d2.shape
    eye = torch.eye(N, device=points.device, dtype=torch.bool).unsqueeze(0)
    d2  = d2.masked_fill(eye, float('inf'))
    knn = torch.topk(d2, k=k, dim=-1, largest=False)[0]
    return torch.exp(-knn / (h * h)).mean()


def project_consistency_loss(pred_world, gt_world, P_ap, P_lp):
    """2-D Chamfer between projected predicted and GT point clouds.
    Penalises predictions that look wrong in the X-ray views.
    Kept but with weight W_PROJ=0.02 (was 0.15) — acts as a geometric anchor,
    not the primary training signal.
    """
    uvp_ap, _, _ = project_points(P_ap, pred_world)
    uvg_ap, _, _ = project_points(P_ap, gt_world)
    uvp_lp, _, _ = project_points(P_lp, pred_world)
    uvg_lp, _, _ = project_points(P_lp, gt_world)
    return chamfer_loss(uvp_ap, uvg_ap) + chamfer_loss(uvp_lp, uvg_lp)


def dice_loss_from_logits(logits, targets, eps=1e-6):
    probs   = torch.sigmoid(logits).reshape(logits.shape[0], -1)
    targets = targets.reshape(targets.shape[0], -1)
    inter   = (probs * targets).sum(dim=1)
    denom   = probs.sum(dim=1) + targets.sum(dim=1)
    return 1.0 - ((2 * inter + eps) / (denom + eps)).mean()


def total_loss(output, batch, current_epoch=0):
    """Full training loss with per-component breakdown for logging.

    W_PROJ is linearly warmed down from 0.10 → 0.02 over the first 30 epochs
    so early training benefits from the 2-D supervision signal before the
    3-D Chamfer takes full control.  After epoch 30 it stays at 0.02.
    """
    l_ch   = chamfer_loss(output['pred_world'], batch['gt_ppc_world'])
    l_rep  = repulsion_loss(output['pred_local'])
    l_proj = project_consistency_loss(
        output['pred_world'], batch['gt_ppc_world'],
        batch['P_ap'], batch['P_lp'])
    l_aux  = (F.binary_cross_entropy_with_logits(
                  output['aux_occ_logits'].squeeze(1), batch['aux_occ'])
              + dice_loss_from_logits(
                  output['aux_occ_logits'].squeeze(1), batch['aux_occ']))
    l_off  = torch.stack([a['delta'].abs().mean() for a in output['stage_aux']]).mean()

    # Warm-down projection weight: 0.10 → 0.02 over epochs 0–30
    w_proj_eff = W_PROJ + max(0.0, (30 - current_epoch) / 30.0) * (0.10 - W_PROJ)

    loss = (W_CHAMFER  * l_ch
          + W_REPULSION * l_rep
          + w_proj_eff  * l_proj
          + W_AUX_OCC   * l_aux
          + W_OFFSET    * l_off)

    breakdown = {
        'total':      float(loss.detach().cpu()),
        'chamfer':    float(l_ch.detach().cpu()),
        'repulsion':  float(l_rep.detach().cpu()),
        'proj':       float(l_proj.detach().cpu()),
        'aux_occ':    float(l_aux.detach().cpu()),
        'offset':     float(l_off.detach().cpu()),
        'w_proj_eff': float(w_proj_eff),
    }
    return loss, breakdown


print('Loss functions defined. Sanity check:')
_p  = torch.randn(2, 5120, 3)
_g  = torch.randn(2, 5120, 3)
_ch = chamfer_loss(_p, _g)
_re = repulsion_loss(_p)
print(f'  chamfer (random) = {_ch:.4f}')
print(f'  repulsion        = {_re:.6f}')
del _p, _g, _ch, _re


In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, best_val_loss, history):
    state = {
        'model':            model.state_dict(),
        'optimizer':        optimizer.state_dict(),
        'scheduler':        scheduler.state_dict() if scheduler else None,
        'scaler':           scaler.state_dict() if scaler else None,
        'epoch':            epoch,
        'best_val_loss':    best_val_loss,
        'training_history': history,
        'config':           {'version': 'v1_fixed_hybrid'},
        'timestamp':        datetime.utcnow().isoformat(),
    }
    tmp = path.with_suffix('.pth.tmp')
    torch.save(state, tmp)
    tmp.replace(path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None):
    state = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(state['model'])
    if optimizer  and state.get('optimizer')  is not None: optimizer.load_state_dict(state['optimizer'])
    if scheduler  and state.get('scheduler')  is not None: scheduler.load_state_dict(state['scheduler'])
    if scaler     and state.get('scaler')     is not None: scaler.load_state_dict(state['scaler'])
    start   = state['epoch'] + 1
    best    = state.get('best_val_loss', float('inf'))
    history = state.get('training_history', [])
    print(f'  Resumed from epoch {start}  |  best_val_loss so far: {best:.4f}')
    return start, best, history


def collate_keep_strings(batch):
    out = {}
    for k in batch[0]:
        vals = [b[k] for b in batch]
        out[k] = torch.stack(vals, 0) if isinstance(vals[0], torch.Tensor) else vals
    return out


def move_to_device(batch):
    return {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v)
            for k, v in batch.items()}


In [10]:
def chamfer_metrics_np(pred, gt):
    """Full per-patient metrics in mm (CPU numpy)."""
    pred = np.asarray(pred, np.float32)
    gt   = np.asarray(gt,   np.float32)
    # (N,M) distance matrix — O(N*M) but fine for 5120 pts
    d    = np.linalg.norm(pred[:, None] - gt[None], axis=-1)
    d_pg = d.min(axis=1)   # pred→GT
    d_gp = d.min(axis=0)   # GT→pred
    def fscore(th):
        p = (d_pg <= th).mean()
        r = (d_gp <= th).mean()
        return 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return {
        'chamfer_mm':  float(0.5 * (d_pg.mean() + d_gp.mean())),
        'fscore_1mm':  float(fscore(1.0)),
        'fscore_2mm':  float(fscore(2.0)),
        'fscore_5mm':  float(fscore(5.0)),
        'hausdorff_95': float(max(np.percentile(d_pg, 95), np.percentile(d_gp, 95))),
    }


@torch.no_grad()
def evaluate(model, loader, max_samples=MAX_EVAL_SAMPLES, current_epoch=0):
    """Evaluate on up to max_samples patients.
    max_samples=103 → full val set every epoch.
    """
    model.eval()
    acc     = {k: 0.0 for k in ('total','chamfer','repulsion','proj','aux_occ','offset')}
    n_bat   = 0
    metrics = []
    n_done  = 0

    for batch in loader:
        batch  = move_to_device(batch)
        output = model(batch['drr_ap'], batch['drr_lp'],
                       batch['P_ap'],  batch['P_lp'],
                       batch['center'], batch['scale'])
        _, bd  = total_loss(output, batch, current_epoch)
        for k in acc:
            acc[k] += bd[k]
        n_bat += 1

        if n_done < max_samples:
            pred = output['pred_world'].cpu().numpy()
            gt   = batch['gt_ppc_world'].cpu().numpy()
            for b in range(pred.shape[0]):
                if n_done >= max_samples:
                    break
                metrics.append(chamfer_metrics_np(pred[b], gt[b]))
                n_done += 1

    result = {k: acc[k] / max(1, n_bat) for k in acc}
    if metrics:
        result.update({
            'mean_mm':      float(np.mean([m['chamfer_mm']  for m in metrics])),
            'std_mm':       float(np.std( [m['chamfer_mm']  for m in metrics])),
            'fscore_1mm':   float(np.mean([m['fscore_1mm']  for m in metrics])),
            'fscore_2mm':   float(np.mean([m['fscore_2mm']  for m in metrics])),
            'fscore_5mm':   float(np.mean([m['fscore_5mm']  for m in metrics])),
            'hausdorff_95': float(np.mean([m['hausdorff_95'] for m in metrics])),
            'n_patients':   n_done,
        })
    else:
        result.update(dict(mean_mm=float('inf'), std_mm=0.0,
                           fscore_1mm=0.0, fscore_2mm=0.0,
                           fscore_5mm=0.0, hausdorff_95=float('inf'),
                           n_patients=0))
    return result


In [ ]:
train_ds = LumbarDatasetv1(train_ids, augment=True)
val_ds   = LumbarDatasetv1(val_ids,   augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_keep_strings, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_keep_strings, persistent_workers=True)

print(f'Train: {len(train_ds)} patients  →  {len(train_loader)} batches/epoch')
print(f'Val  : {len(val_ds)}   patients  →  {len(val_loader)}   batches/epoch')


In [ ]:
model     = PPCNetv1().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == 'cuda')

# Cosine schedule with linear warmup
warmup_steps = max(1, WARMUP_EPOCHS * len(train_loader))
total_steps  = max(1, EPOCHS * len(train_loader))

def lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

start_epoch    = 0
best_val_loss  = float('inf')
best_chamfer   = float('inf')
history        = []

if CKPT_PATH.exists():
    print('Resuming from checkpoint …')
    start_epoch, best_val_loss, history = load_checkpoint(
        CKPT_PATH, model, optimizer, scheduler, scaler)
    best_chamfer = min((r['val'].get('mean_mm', float('inf')) for r in history), default=float('inf'))
else:
    print('No checkpoint found — starting fresh.')

print(f'Model params  : {count_params(model)/1e6:.1f} M')
print(f'Start epoch   : {start_epoch + 1}')
print(f'Best val loss : {best_val_loss:.4f}')
print(f'Best Chamfer  : {best_chamfer:.3f} mm')


In [ ]:
for epoch in range(start_epoch, EPOCHS):
    model.train()
    acc = {k: 0.0 for k in ('total','chamfer','repulsion','proj','aux_occ','offset')}

    for bi, batch in enumerate(train_loader, start=1):
        batch = move_to_device(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP and device.type == 'cuda'):
            output = model(batch['drr_ap'], batch['drr_lp'],
                           batch['P_ap'],  batch['P_lp'],
                           batch['center'], batch['scale'])
            loss, bd = total_loss(output, batch, current_epoch=epoch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        for k in acc:
            acc[k] += bd[k]

        if bi % 100 == 0 or bi == len(train_loader):
            lr_now = optimizer.param_groups[0]['lr']
            print(f"  [Ep {epoch+1:3d}/{EPOCHS}] {bi:3d}/{len(train_loader)} "
                  f"loss={bd['total']:.4f}  "
                  f"ch={bd['chamfer']:.4f}  "
                  f"proj={bd['proj']:.4f}  "
                  f"aux={bd['aux_occ']:.4f}  "
                  f"w_proj={bd['w_proj_eff']:.3f}  "
                  f"lr={lr_now:.2e}")

    train_stats = {k: acc[k] / max(1, len(train_loader)) for k in acc}
    val_stats   = evaluate(model, val_loader, max_samples=MAX_EVAL_SAMPLES,
                           current_epoch=epoch)

    rec = {'epoch': epoch + 1, 'train': train_stats, 'val': val_stats,
           'lr': optimizer.param_groups[0]['lr']}
    history.append(rec)
    append_training_log(TRAINING_LOG, rec)

    # Always save latest (for resume)
    save_checkpoint(CKPT_PATH, model, optimizer, scheduler, scaler,
                    epoch, best_val_loss, history)

    # ── print epoch summary ──────────────────────────────────────────────────
    print(f"[Epoch {epoch+1:3d}/{EPOCHS}] "
          f"train={train_stats['total']:.4f}  "
          f"val={val_stats['total']:.4f}")
    print(f"  Chamfer={val_stats['mean_mm']:.3f}±{val_stats['std_mm']:.3f} mm  "
          f"F@1={val_stats['fscore_1mm']:.3f}  "
          f"F@2={val_stats['fscore_2mm']:.3f}  "
          f"F@5={val_stats['fscore_5mm']:.3f}  "
          f"HD95={val_stats['hausdorff_95']:.3f} mm  "
          f"({val_stats['n_patients']} patients)")

    # Save best by val_loss (composite)
    if val_stats['total'] < best_val_loss:
        best_val_loss = val_stats['total']
        save_checkpoint(BEST_CKPT_PATH, model, optimizer, scheduler, scaler,
                        epoch, best_val_loss, history)
        print(f"  ✓ New best val_loss: {best_val_loss:.4f}")

    # Also track best Chamfer separately
    if val_stats['mean_mm'] < best_chamfer:
        best_chamfer = val_stats['mean_mm']
        best_ch_path = CKPT_DIR / 'best_chamfer_checkpoint.pth'
        save_checkpoint(best_ch_path, model, optimizer, scheduler, scaler,
                        epoch, best_val_loss, history)
        print(f"  ✓ New best Chamfer: {best_chamfer:.3f} mm")

print('\nTraining complete.')
print(f'Best val_loss : {best_val_loss:.4f}')
print(f'Best Chamfer  : {best_chamfer:.3f} mm')


In [ ]:
# ── Load best checkpoint and evaluate on held-out test set ──────────────────
print('Loading best Chamfer checkpoint for test evaluation …')
best_ch_path = CKPT_DIR / 'best_chamfer_checkpoint.pth'
ckpt_to_use  = best_ch_path if best_ch_path.exists() else BEST_CKPT_PATH

state = torch.load(ckpt_to_use, map_location=device, weights_only=False)
model_eval = PPCNetv1().to(device)
model_eval.load_state_dict(state['model'])
model_eval.eval()
print(f'  Loaded epoch {state["epoch"] + 1}')

test_ds     = LumbarDatasetv1(test_ids, augment=False)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         collate_fn=collate_keep_strings)

all_metrics = []
with torch.no_grad():
    for batch in test_loader:
        batch  = move_to_device(batch)
        output = model_eval(batch['drr_ap'], batch['drr_lp'],
                            batch['P_ap'],  batch['P_lp'],
                            batch['center'], batch['scale'])
        pred = output['pred_world'].cpu().numpy()
        gt   = batch['gt_ppc_world'].cpu().numpy()
        pids = batch['patient_id']
        for b in range(pred.shape[0]):
            m = chamfer_metrics_np(pred[b], gt[b])
            m['patient_id'] = pids[b]
            all_metrics.append(m)
            # Save output point cloud
            save_vtk_points(pred[b], RESULTS_DIR / f'{pids[b]}_pred.vtk')

print(f'\n{"="*60}')
print(f'TEST SET RESULTS  ({len(all_metrics)} patients)')
print(f'{"="*60}')
keys = ['chamfer_mm', 'fscore_1mm', 'fscore_2mm', 'fscore_5mm', 'hausdorff_95']
labels = ['Chamfer (mm)', 'F@1mm', 'F@2mm', 'F@5mm', 'HD95 (mm)']
for k, lbl in zip(keys, labels):
    vals = [m[k] for m in all_metrics]
    print(f'  {lbl:<16}  mean={np.mean(vals):.3f}  std={np.std(vals):.3f}  '
          f'median={np.median(vals):.3f}  '
          f'p25={np.percentile(vals,25):.3f}  p75={np.percentile(vals,75):.3f}')

# Save full results table
import csv
csv_path = RESULTS_DIR / 'test_results.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['patient_id'] + keys)
    writer.writeheader()
    writer.writerows(all_metrics)
print(f'\nPer-patient results saved to {csv_path}')


In [ ]:
# Plot training history (runs in-notebook — no display needed on cluster,
# saves PNGs to results dir for later inspection)
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    epochs  = [r['epoch']              for r in history]
    tr_loss = [r['train']['total']     for r in history]
    va_loss = [r['val']['total']       for r in history]
    chamfer = [r['val'].get('mean_mm', float('nan')) for r in history]
    f2      = [r['val'].get('fscore_2mm', float('nan')) for r in history]
    f5      = [r['val'].get('fscore_5mm', float('nan')) for r in history]
    hd95    = [r['val'].get('hausdorff_95', float('nan')) for r in history]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('PPC v1 — Training History', fontsize=13)

    axes[0,0].plot(epochs, tr_loss, label='train', alpha=0.7)
    axes[0,0].plot(epochs, va_loss, label='val',   alpha=0.9)
    axes[0,0].set_title('Total loss'); axes[0,0].legend(); axes[0,0].set_xlabel('epoch')

    axes[0,1].plot(epochs, chamfer, color='steelblue')
    axes[0,1].axhline(2.0, ls='--', color='red',   alpha=0.5, label='2 mm target')
    axes[0,1].set_title('Chamfer distance (mm)'); axes[0,1].legend(); axes[0,1].set_xlabel('epoch')

    axes[1,0].plot(epochs, f2, label='F@2mm', color='darkorange')
    axes[1,0].plot(epochs, f5, label='F@5mm', color='forestgreen')
    axes[1,0].axhline(0.60, ls='--', color='darkorange', alpha=0.4, label='F@2 target')
    axes[1,0].axhline(0.95, ls='--', color='forestgreen', alpha=0.4, label='F@5 target')
    axes[1,0].set_title('F-score'); axes[1,0].legend(); axes[1,0].set_xlabel('epoch')

    axes[1,1].plot(epochs, hd95, color='crimson')
    axes[1,1].axhline(5.0, ls='--', color='crimson', alpha=0.5, label='5 mm target')
    axes[1,1].set_title('HD95 (mm)'); axes[1,1].legend(); axes[1,1].set_xlabel('epoch')

    plt.tight_layout()
    plot_path = RESULTS_DIR / 'training_history.png'
    plt.savefig(plot_path, dpi=150)
    plt.close()
    print(f'Plot saved to {plot_path}')
except ImportError:
    print('matplotlib not available — skipping plot')


In [ ]:
def predict_ppc(drr_ap_path, drr_lp_path, p_ap_path, p_lp_path,
                output_path,
                checkpoint_path=None,
                center_mm=None,
                scale_mm=None):
    """Run inference on a new patient at test time.

    center_mm and scale_mm:
        If you have the patient's CT-derived bounding box, pass the centroid and
        per-axis half-extent here for best results.  If not, the defaults
        (lumbar-average values) will be used — results will still be in the
        correct projection-matrix coordinate space.

        Default center=(0,0,0) is correct only if the projection matrices were
        generated with the origin at the patient centroid.  If your pipeline
        places the origin elsewhere, provide the real centroid.
    """
    if checkpoint_path is None:
        best_ch = CKPT_DIR / 'best_chamfer_checkpoint.pth'
        checkpoint_path = best_ch if best_ch.exists() else BEST_CKPT_PATH

    state = torch.load(checkpoint_path, map_location=device, weights_only=False)
    net   = PPCNetv1().to(device)
    net.load_state_dict(state['model'])
    net.eval()
    print(f'Loaded checkpoint (epoch {state["epoch"] + 1})')

    img_norm = transforms.Normalize(mean=[0.485], std=[0.229])

    def _load(path):
        img = load_drr_image(path)
        return img_norm(torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float()).to(device)

    drr_ap_t = _load(drr_ap_path)
    drr_lp_t = _load(drr_lp_path)
    P_ap_t   = torch.from_numpy(load_projection_matrix(p_ap_path)).unsqueeze(0).float().to(device)
    P_lp_t   = torch.from_numpy(load_projection_matrix(p_lp_path)).unsqueeze(0).float().to(device)

    # Sensible lumbar-average defaults
    if center_mm is None:
        center_mm = [0.0, 0.0, 0.0]
    if scale_mm is None:
        scale_mm  = [80.0, 80.0, 130.0]

    center_t = torch.tensor([center_mm], dtype=torch.float32, device=device)
    scale_t  = torch.tensor([scale_mm],  dtype=torch.float32, device=device)

    with torch.no_grad():
        out   = net(drr_ap_t, drr_lp_t, P_ap_t, P_lp_t, center_t, scale_t)
        pred  = out['pred_world'].squeeze(0).cpu().numpy()

    save_vtk_points(pred, output_path)
    print(f'Saved {len(pred)} points → {output_path}')
    return pred


print('predict_ppc() ready.')
print('Usage:')
print('  pts = predict_ppc(')
print('      drr_ap_path="AP_0/drr_AP_0.png",')
print('      drr_lp_path="LP_90/drr_LP_90.png",')
print('      p_ap_path  ="AP_0/P_AP_0.txt",')
print('      p_lp_path  ="LP_90/P_LP_90.txt",')
print('      output_path="pred_ppc.vtk",')
print('      # optional: center_mm=[cx,cy,cz], scale_mm=[sx,sy,sz]')
print('  )')


## Summary of all v1 changes

### Loss weights
| Weight | v0 | v1 | Reason |
|---|---|---|---|
| `W_CHAMFER` | 1.0 | 1.0 | unchanged — primary signal |
| `W_PROJ` | 0.15 | **0.02** (warm-down from 0.10) | proj loss weighted 3.6× Chamfer in v0; now a minor anchor |
| `W_REPULSION` | 0.03 | **0.01** | repulsion `h` also fixed |
| `W_AUX_OCC` | 0.10 | **0.05** | pure regulariser |
| `W_OFFSET` | 0.002 | **0.0005** | let refinement stages move freely |

### Architecture
| Component | v0 | v1 |
|---|---|---|
| `FeatureLift` | expand 2-D slice, zero depth variation | + learned `depth_embed` parameter |
| `QueryInitializer` global offset | `±0.15 × tanh` | `±0.20 × tanh` |
| `RefinementStage` delta | `±0.15 × tanh` + clamp each stage | `±0.25 × tanh`, **clamp only at final output** |

### Data
| | v0 | v1 |
|---|---|---|
| Scale computation | `max(extent×0.65, [40,40,60])` | `extent×0.55 + [20,20,30]` — always proportional |
| Augmentation | intensity jitter only | + consistent horizontal flip of DRRs **and** projection matrices |
| `MAX_EVAL_SAMPLES` | 20 | **103** (full val set) |

### Expected results (lumbar, 1037 patients)
| Metric | v0 best (42 ep) | v1 target |
|---|---|---|
| Chamfer | ~2.4 mm | **< 2.0 mm** |
| F@2mm | ~0.46 | **> 0.60** |
| F@5mm | ~0.95 | **> 0.97** |
| HD95 | ~5.0 mm | **< 4.5 mm** |

These targets are consistent with published biplanar X-ray → point cloud results
(e.g. Gu et al. 2023, StatFormer, and related vertebra shape reconstruction work).
